In [9]:
# Core RAG Frameworks

!pip install langchain==1.3.13
!pip install langchain-core==1.5.4
!pip install langchain-community==0.4.2
!pip install langchain-openai
!pip install langchain-text-splitters
!pip install langgraph
!pip install faiss-cpu
!pip install tiktoken
!pip install openai

In [10]:
import os
# Import the Python SDK for Colab secrets
from google.colab import userdata

# Fetch the API key from Colab secrets
openai_api_key = userdata.get('OPENAI_APIKEY')

# Set up Environmental Variables
os.environ["OPENAI_API_KEY"] = openai_api_key

In [11]:
# Import all depedencies

from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import FAISS
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain.agents import create_agent
# NEW IMPORT
from langchain_core.tools import create_retriever_tool

In [12]:
# ==========================================
# Step 1: Prepare Knowledge Base (Documents)
# ==========================================
# For demonstration, creating a dummy document

sample_text = """
ACME Corp Remote Work Policy (2026)

1. Employees may work remotely up to 3 days per week.
2. Annual home office stipend is $500.
3. Core working hours are 10AM-4PM EST.
4. Internet reimbursement up to $70/month.
"""

with open("company_policy.txt", "w", encoding="utf-8") as f:
    f.write(sample_text)



In [13]:
# Load the documents
loader = TextLoader("company_policy.txt")
documents = loader.load()

# Split documents into chunks
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
docs = text_splitter.split_documents(documents)

In [14]:
# Embed and store in Vector Database (FAISS)
embeddings = OpenAIEmbeddings()
vectorstore = FAISS.from_documents(docs, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

In [15]:
# ==========================================
# Step 2: Create the RAG Retriever Tool
# ==========================================
# This wraps the retriever into a tool that the Agent can call on demand.
retriever_tool = create_retriever_tool(
    retriever,
    name="company_policy_search",
    description="Searches and retrieves relevant policy information regarding ACME Corp."
)


In [16]:
# ==========================================
# Step 3: Initialize LLM and Prompt
# ==========================================
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

instruction_prompt="""
You are an HR assistant.

Always use the company_policy_search tool
before answering HR policy questions.

If the answer isn't available,
say that the policy could not be found.
"""




In [17]:
# ==========================================
# Step 4: Assemble and Run the Agent
# ==========================================
# Create Agent

agent = create_agent(
    model=llm,
    tools=[retriever_tool],
    system_prompt=instruction_prompt
    )



In [33]:
# Example 1: Question requiring Knowledge Base lookup
response_1 = agent.invoke({"messages": [{"role": "user","content": "How much is the internet reimbursement?"}]})
print("\n--- Final Answer ---")
print(response_1["messages"][2].content)

# Example 2: Question NOT requiring RAG (Agent skips tool execution)
response_2 = agent.invoke({"messages": [{"role": "user","content": "What is 15 multiplied by 12?"}]})
print("\n--- Final Answer ---")
print(response_2["messages"][1].content)


--- Final Answer ---
ACME Corp Remote Work Policy (2026)

1. Employees may work remotely up to 3 days per week.
2. Annual home office stipend is $500.
3. Core working hours are 10AM-4PM EST.
4. Internet reimbursement up to $70/month.

--- Final Answer ---
15 multiplied by 12 is 180.
